# Όραση Υπολογιστών

In [211]:
%pip install opencv-python numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


Εγκαθιστούμε τις βασικές βιβλιοθήκες που χρειάζεται η εφαρμογή. Η OpenCV χρησιμοποιείται για την κάμερα, την επεξεργασία εικόνας, τα φίλτρα ακμών και τα contours. Η NumPy χρησιμοποιείται για αριθμητικές πράξεις πάνω στις εικόνες.

In [212]:
import cv2
import numpy as np
import time
import os
import csv

Το `cv2` είναι η `OpenCV`. Το `numpy` χρησιμοποιείται για πράξεις πάνω σε πίνακες εικόνων. Το `time` χρησιμοποιείται για τον υπολογισμό των FPS, δηλαδή των καρέ ανά δευτερόλεπτο.

In [213]:
CAMERA_INDEX = 0
FRAME_WIDTH = 800

MIN_CONTOUR_AREA = 1800
MIN_BOX_AREA = 3500
MAX_OBJECTS_TO_DRAW = 6

# Sunglasses thresholds
SUNGLASSES_MIN_ASPECT = 1.75
SUNGLASSES_MIN_LONG_RATIO = 1.75
SUNGLASSES_MAX_EXTENT = 0.80
SUNGLASSES_MAX_SOLIDITY = 0.95

# Book / Notebook thresholds
BOOK_MIN_LONG_RATIO = 1.20
BOOK_MAX_LONG_RATIO = 3.80

# Box thresholds
BOX_MIN_ASPECT = 0.80
BOX_MAX_ASPECT = 1.25
BOX_MAX_LONG_RATIO = 1.25

BOOK_MIN_EXTENT = 0.45
BOX_MIN_EXTENT = 0.30

COMPACT_MIN_SOLIDITY = 0.70

EXPERIMENT_DIR = os.path.join(os.path.expanduser("~"), "Desktop", "cv_experiment_results")

Το `CAMERA_INDEX = 0` σημαίνει ότι χρησιμοποιείται η βασική κάμερα του υπολογιστή.

Το `FRAME_WIDTH = 800` κάνει resize το βίντεο για να τρέχει πιο γρήγορα η εφαρμογή.

Τα `MIN_CONTOUR_AREA` και `MIN_BOX_AREA` χρησιμοποιούνται για να αγνοούμε πολύ μικρά contours, τα οποία συνήθως είναι θόρυβος.

Το `MAX_OBJECTS_TO_DRAW` περιορίζει πόσα αντικείμενα θα σχεδιάζονται κάθε φορά.

In [214]:
def resize_frame(frame, width=FRAME_WIDTH):
    height, original_width = frame.shape[:2]

    scale = width / original_width
    new_height = int(height * scale)

    resized = cv2.resize(frame, (width, new_height))

    return resized

Η κάμερα μπορεί να επιστρέφει εικόνα σε μεγάλο μέγεθος. Για να είναι πιο γρήγορη η εφαρμογή, κάνουμε resize κάθε frame σε σταθερό πλάτος. Κρατάμε όμως το σωστό aspect ratio, ώστε η εικόνα να μη φαίνεται παραμορφωμένη.

In [215]:
def preprocess_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    return blurred

Πρώτα μετατρέπουμε την εικόνα σε grayscale, γιατί οι μέθοδοι ανίχνευσης ακμών βασίζονται κυρίως στις μεταβολές φωτεινότητας.
Μετά εφαρμόζουμε Gaussian Blur για να μειωθεί ο θόρυβος. Αυτό βοηθάει ώστε να μη δημιουργούνται πολλές ψεύτικες ακμές.

In [216]:
def normalize_to_uint8(image):
    normalized = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    normalized = normalized.astype(np.uint8)

    return normalized

Οι μέθοδοι `Sobel` και `Laplacian` μπορεί να παράγουν τιμές εκτός του εύρους 0–255. Η συνάρτηση αυτή μετατρέπει το αποτέλεσμα σε εικόνα τύπου `uint8`, ώστε να μπορεί να εμφανιστεί σωστά και να χρησιμοποιηθεί για thresholding.

In [217]:
def apply_edge_detection(gray, method):
    if method == "canny":
        edges = cv2.Canny(gray, 50, 150)
        return edges

    elif method == "sobel":
        grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

        magnitude = cv2.magnitude(grad_x, grad_y)
        magnitude = normalize_to_uint8(magnitude)

        _, edges = cv2.threshold(
            magnitude,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        return edges

    elif method == "laplacian":
        lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
        lap = np.absolute(lap)
        lap = normalize_to_uint8(lap)

        _, edges = cv2.threshold(
            lap,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        return edges

    else:
        raise ValueError("Άγνωστη μέθοδος edge detection")

Η συνάρτηση αυτή εφαρμόζει μία από τις τρεις μεθόδους ανίχνευσης ακμών.

Ο `Canny` παράγει συνήθως λεπτές και καθαρές ακμές.
Ο `Sobel` υπολογίζει τις μεταβολές φωτεινότητας στους άξονες x και y.
Ο `Laplacian` βασίζεται στη δεύτερη παράγωγο και μπορεί να εντοπίζει ακμές προς πολλές κατευθύνσεις, αλλά είναι πιο ευαίσθητος στον θόρυβο.

In [218]:
def clean_edges(edges):
    kernel = np.ones((3, 3), np.uint8)

    closed = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=1
    )

    dilated = cv2.dilate(
        closed,
        kernel,
        iterations=1
    )

    return dilated

Μετά την ανίχνευση ακμών, οι γραμμές μπορεί να έχουν μικρά κενά.
Το `MORPH_CLOSE` βοηθάει να κλείσουν αυτά τα κενά.
Το `dilate` παχαίνει λίγο τις ακμές, ώστε τα contours να βγαίνουν πιο ενωμένα και πιο σταθερά.

In [219]:
def classify_object(contour, frame_area):
    area = cv2.contourArea(contour)

    if area < MIN_CONTOUR_AREA:
        return None

    x, y, w, h = cv2.boundingRect(contour)
    box_area = w * h

    if box_area < MIN_BOX_AREA:
        return None

    if box_area > 0.45 * frame_area:
        return None

    if w < 40 or h < 40:
        return None

    aspect_ratio = w / float(h)
    long_ratio = max(w, h) / float(min(w, h))
    extent = area / float(box_area)

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)

    if hull_area == 0:
        return None

    solidity = area / float(hull_area)

    perimeter = cv2.arcLength(contour, True)

    if perimeter == 0:
        return None

    approx = cv2.approxPolyDP(contour, 0.04 * perimeter, True)
    vertices = len(approx)

    features = {
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "area": area,
        "box_area": box_area,
        "aspect_ratio": aspect_ratio,
        "long_ratio": long_ratio,
        "extent": extent,
        "solidity": solidity,
        "vertices": vertices
    }

    matches = []

    # 1. Sunglasses
    # Τα γυαλιά είναι φαρδύ αντικείμενο, αλλά συνήθως λιγότερο συμπαγές
    # από ένα βιβλίο ή ένα κουτί. Χρησιμοποιούμε και aspect_ratio και long_ratio,
    # ώστε να αναγνωρίζονται ακόμη και αν έχουν μικρή κλίση στην κάμερα.
    if (
        aspect_ratio >= SUNGLASSES_MIN_ASPECT
        and long_ratio >= SUNGLASSES_MIN_LONG_RATIO
        and extent <= SUNGLASSES_MAX_EXTENT
        and solidity <= SUNGLASSES_MAX_SOLIDITY
        and 4 <= vertices <= 60
    ):
        matches.append(("Sunglasses", (255, 0, 255)))

    # 2. Book / Notebook
    # Στενόμακρο αντικείμενο. Το όριο ξεκινά από 1.20,
    # ώστε να μη μπερδεύεται εύκολα το τετράδιο με το κουτί.
    if (
        BOOK_MIN_LONG_RATIO <= long_ratio <= BOOK_MAX_LONG_RATIO
        and extent >= BOOK_MIN_EXTENT
        and solidity >= COMPACT_MIN_SOLIDITY
        and 4 <= vertices <= 25
    ):
        matches.append(("Book/Notebook", (255, 0, 0)))

    # 3. Box
    # Πιο τετράγωνο αντικείμενο. Το long_ratio πρέπει να είναι κοντά στο 1.
    if (
        BOX_MIN_ASPECT <= aspect_ratio <= BOX_MAX_ASPECT
        and long_ratio <= BOX_MAX_LONG_RATIO
        and extent >= BOX_MIN_EXTENT
        and 4 <= vertices <= 30
    ):
        matches.append(("Box", (0, 165, 255)))

    # Αν υπάρχει ακριβώς μία κατηγορία, επιστρέφουμε αυτή.
    if len(matches) == 1:
        label, color = matches[0]
        return label, color, features

    if len(matches) > 1:
        labels = [label for label, color in matches]

        # Αν ένα αντικείμενο είναι πολύ φαρδύ και λιγότερο συμπαγές,
        # το θεωρούμε Sunglasses και όχι Book/Notebook.
        if (
            "Sunglasses" in labels
            and long_ratio >= SUNGLASSES_MIN_LONG_RATIO
            and extent <= SUNGLASSES_MAX_EXTENT
            and solidity <= SUNGLASSES_MAX_SOLIDITY
        ):
            return "Sunglasses", (255, 0, 255), features

        if "Book/Notebook" in labels:
            return "Book/Notebook", (255, 0, 0), features

        if "Box" in labels:
            return "Box", (0, 165, 255), features

    return None

Εδώ γίνεται η απλή αναγνώριση των 3 αντικειμένων. Δεν χρησιμοποιείται τεχνητή νοημοσύνη ή εκπαιδευμένο μοντέλο. Η κατηγοριοποίηση γίνεται με απλούς κανόνες πάνω στα γεωμετρικά χαρακτηριστικά των contours.

Τα γυαλιά θεωρούνται φαρδύ και χαμηλό αντικείμενο.
Το βιβλίο/τετράδιο θεωρείται ορθογώνιο αντικείμενο με μία πλευρά αρκετά μεγαλύτερη από την άλλη.
Το κουτί θεωρείται πιο τετράγωνο και συμπαγές αντικείμενο.

In [220]:
def draw_result(frame, contour, label, color, features):
    x = features["x"]
    y = features["y"]
    w = features["w"]
    h = features["h"]

    cv2.drawContours(frame, [contour], -1, color, 2)

    cv2.rectangle(
        frame,
        (x, y),
        (x + w, y + h),
        color,
        2
    )

    cv2.putText(
        frame,
        label,
        (x, max(y - 10, 25)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        color,
        2
    )

Για κάθε αντικείμενο που αναγνωρίζεται, σχεδιάζεται το contour, το bounding box και η ετικέτα του αντικειμένου.
Χρησιμοποιούμε διαφορετικό χρώμα για κάθε κατηγορία, ώστε να ξεχωρίζουν οπτικά τα αποτελέσματα.

In [221]:
def get_detected_labels(detected_objects):
    labels = [label for _, label, _, _ in detected_objects]

    required_labels = ["Sunglasses", "Book/Notebook", "Box"]

    summary = []

    for required_label in required_labels:
        if required_label in labels:
            summary.append(f"{required_label}: YES")
        else:
            summary.append(f"{required_label}: NO")

    return " | ".join(summary)

In [222]:
def print_measurements(detected_objects, method, fps):
    print("\n--- Μετρήσεις αντικειμένων ---")
    print(f"Μέθοδος: {method.upper()}")
    print(f"FPS: {fps:.1f}")
    print(f"Σύνολο αντικειμένων που αναγνωρίστηκαν: {len(detected_objects)}")

    if len(detected_objects) == 0:
        print("Δεν αναγνωρίστηκε κάποιο αντικείμενο.")
        print("-----------------------------\n")
        return

    for i, (contour, label, color, features) in enumerate(
        detected_objects[:MAX_OBJECTS_TO_DRAW],
        start=1
    ):
        print(
            f"{i}. {label} | "
            f"aspect_ratio={features['aspect_ratio']:.2f}, "
            f"long_ratio={features['long_ratio']:.2f}, "
            f"extent={features['extent']:.2f}, "
            f"solidity={features['solidity']:.2f}, "
            f"vertices={features['vertices']}, "
            f"area={features['area']:.1f}, "
            f"box_area={features['box_area']}, "
            f"w={features['w']}, "
            f"h={features['h']}"
        )

    print("-----------------------------\n")

In [223]:
def save_experiment_results(output, cleaned_edges, detected_objects, method, fps):
    os.makedirs(EXPERIMENT_DIR, exist_ok=True)

    timestamp = time.strftime("%Y%m%d_%H%M%S")

    output_filename = os.path.join(
        EXPERIMENT_DIR,
        f"{timestamp}_{method}_output.png"
    )

    edges_filename = os.path.join(
        EXPERIMENT_DIR,
        f"{timestamp}_{method}_edges.png"
    )

    csv_filename = os.path.join(
        EXPERIMENT_DIR,
        "measurements.csv"
    )

    output_saved = cv2.imwrite(output_filename, output)
    edges_saved = cv2.imwrite(edges_filename, cleaned_edges)

    if not output_saved:
        print("Σφάλμα: Δεν αποθηκεύτηκε το output screenshot.")
        print(f"Διαδρομή: {output_filename}")
        return

    if not edges_saved:
        print("Σφάλμα: Δεν αποθηκεύτηκε το edges screenshot.")
        print(f"Διαδρομή: {edges_filename}")
        return

    file_exists = os.path.isfile(csv_filename)

    labels = [label for _, label, _, _ in detected_objects]

    all_three_detected = (
        "Sunglasses" in labels
        and "Book/Notebook" in labels
        and "Box" in labels
    )

    with open(csv_filename, mode="a", newline="", encoding="utf-8") as file:
        fieldnames = [
            "timestamp",
            "method",
            "fps",
            "total_detected_objects",
            "all_three_detected",
            "object_label",
            "aspect_ratio",
            "long_ratio",
            "extent",
            "solidity",
            "vertices",
            "area",
            "box_area",
            "x",
            "y",
            "w",
            "h",
            "output_screenshot",
            "edges_screenshot"
        ]

        writer = csv.DictWriter(file, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()

        for contour, label, color, features in detected_objects[:MAX_OBJECTS_TO_DRAW]:
            writer.writerow({
                "timestamp": timestamp,
                "method": method.upper(),
                "fps": round(fps, 1),
                "total_detected_objects": len(detected_objects),
                "all_three_detected": "YES" if all_three_detected else "NO",
                "object_label": label,
                "aspect_ratio": round(features["aspect_ratio"], 3),
                "long_ratio": round(features["long_ratio"], 3),
                "extent": round(features["extent"], 3),
                "solidity": round(features["solidity"], 3),
                "vertices": features["vertices"],
                "area": round(features["area"], 1),
                "box_area": features["box_area"],
                "x": features["x"],
                "y": features["y"],
                "w": features["w"],
                "h": features["h"],
                "output_screenshot": output_filename,
                "edges_screenshot": edges_filename
            })

    print("\nΑποθηκεύτηκαν τα πειραματικά αποτελέσματα!")
    print(f"Φάκελος: {EXPERIMENT_DIR}")
    print(f"Output screenshot: {output_filename}")
    print(f"Edges screenshot: {edges_filename}")
    print(f"CSV μετρήσεων: {csv_filename}")

    if all_three_detected:
        print("Κατάσταση: Αναγνωρίστηκαν και τα 3 αντικείμενα.")
    else:
        print("Κατάσταση: Δεν αναγνωρίστηκαν και τα 3 αντικείμενα.")

    print()

In [224]:
def get_detected_labels(detected_objects):
    labels = [label for _, label, _, _ in detected_objects]

    required_labels = ["Sunglasses", "Book/Notebook", "Box"]

    summary = []

    for required_label in required_labels:
        if required_label in labels:
            summary.append(f"{required_label}: YES")
        else:
            summary.append(f"{required_label}: NO")

    return " | ".join(summary)

In [225]:
def run_realtime_app():
    cap = cv2.VideoCapture(CAMERA_INDEX)

    if not cap.isOpened():
        print("Δεν μπόρεσε να ανοίξει η κάμερα.")
        return

    method = "canny"
    previous_time = time.time()

    window_name = "Real-Time Edge Detection and Object Contours"
    edges_window_name = "Edges"

    print("Η εφαρμογή ξεκίνησε.")
    print("Τοποθέτησε ταυτόχρονα στη σκηνή: γυαλιά, βιβλίο/τετράδιο και κουτί.")
    print("Πάτησε 1 για Canny.")
    print("Πάτησε 2 για Sobel.")
    print("Πάτησε 3 για Laplacian.")
    print("Πάτησε m για εκτύπωση μετρήσεων.")
    print("Πάτησε s για αποθήκευση screenshot και CSV.")
    print("Πάτησε q ή ESC για έξοδο.")

    try:
        while True:
            ret, frame = cap.read()

            if not ret:
                print("Δεν μπόρεσε να διαβαστεί frame από την κάμερα.")
                break

            frame = resize_frame(frame)
            output = frame.copy()

            gray = preprocess_frame(frame)

            edges = apply_edge_detection(gray, method)
            cleaned_edges = clean_edges(edges)

            contours, _ = cv2.findContours(
                cleaned_edges,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            frame_area = frame.shape[0] * frame.shape[1]

            detected_objects = []

            for contour in contours:
                result = classify_object(contour, frame_area)

                if result is not None:
                    label, color, features = result
                    detected_objects.append((contour, label, color, features))

            detected_objects = sorted(
                detected_objects,
                key=lambda item: item[3]["box_area"],
                reverse=True
            )

            for contour, label, color, features in detected_objects[:MAX_OBJECTS_TO_DRAW]:
                draw_result(output, contour, label, color, features)

            current_time = time.time()
            fps = 1 / (current_time - previous_time)
            previous_time = current_time

            detected_summary = get_detected_labels(detected_objects)

            cv2.putText(
                output,
                f"Method: {method.upper()} | FPS: {fps:.1f} | Objects: {len(detected_objects)}",
                (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2
            )

            cv2.putText(
                output,
                detected_summary,
                (15, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 255),
                2
            )

            cv2.putText(
            output,
            "Keys: 1=Canny, 2=Sobel, 3=Laplacian, m=Metrics, s=Save, q/ESC=Exit",
            (15, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 255),
            2
        )

            cv2.imshow(window_name, output)
            cv2.imshow(edges_window_name, cleaned_edges)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("1"):
                method = "canny"
                print("Μέθοδος: Canny")

            elif key == ord("2"):
                method = "sobel"
                print("Μέθοδος: Sobel")

            elif key == ord("3"):
                method = "laplacian"
                print("Μέθοδος: Laplacian")

            elif key == ord("m"):
                print_measurements(detected_objects, method, fps)

            elif key == ord("s") or key == ord("S"):
                print_measurements(detected_objects, method, fps)
                save_experiment_results(
                    output,
                    cleaned_edges,
                    detected_objects,
                    method,
                    fps
                )

            elif key == ord("q") or key == 27:
                print("Τερματισμός εφαρμογής.")
                break

            if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                print("Το παράθυρο έκλεισε.")
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()

        for _ in range(5):
            cv2.waitKey(1)

        print("Η κάμερα απελευθερώθηκε και τα παράθυρα έκλεισαν.")

Αυτή είναι η βασική συνάρτηση της εφαρμογής. Ανοίγει την κάμερα, διαβάζει frames σε πραγματικό χρόνο, εφαρμόζει την επιλεγμένη μέθοδο ανίχνευσης ακμών, βρίσκει contours και προσπαθεί να αναγνωρίσει τα τρία αντικείμενα.

Με τα πλήκτρα:

1-> Canny

2 -> Sobel

3 -> Laplacian

q ή ESC για έξοδο

Στην οθόνη εμφανίζονται η ενεργή μέθοδος, τα FPS και ο αριθμός των αντικειμένων που εντοπίστηκαν.

In [226]:
run_realtime_app()

Η εφαρμογή ξεκίνησε.
Τοποθέτησε ταυτόχρονα στη σκηνή: γυαλιά, βιβλίο/τετράδιο και κουτί.
Πάτησε 1 για Canny.
Πάτησε 2 για Sobel.
Πάτησε 3 για Laplacian.
Πάτησε m για εκτύπωση μετρήσεων.
Πάτησε s για αποθήκευση screenshot και CSV.
Πάτησε q ή ESC για έξοδο.
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Canny
Μέθοδος: Sobel

--- Μετρήσεις αντικειμένων ---
Μέθοδος: SOBEL
FPS: 30.0
Σύνολο αντικειμένων που αναγνωρίστηκαν: 3
1. Book/Notebook | aspect_ratio=0.77, long_ratio=1.30, extent=0.84, solidity=0.98, vertices=4, area=19813.5, box_area=23625, w=135, h=175
2. Box | aspect_ratio=0.98, long_ratio=1.02, extent=0.46, solidity=0.72, vertices=6, area=6446.5, box_area=13923, w=117, h=119
3. Sunglasses | aspect_ratio=2.47, long_ratio=2.47, extent=0.69, solidity=0.83, vertices=6, area=5145.0, box_area=7480, w=136, h=55
-----------------------------


Αποθηκεύτηκαν τα πειρ

Η εφαρμογή υλοποιεί real-time ανίχνευση ακμών με τις μεθόδους Canny, Sobel και Laplacian. 
Μετά την εφαρμογή κάθε φίλτρου, πραγματοποιείται καθαρισμός των ακμών με μορφολογικές πράξεις και στη συνέχεια εξάγονται contours.

Για κάθε contour υπολογίζονται απλά γεωμετρικά χαρακτηριστικά, όπως το εμβαδόν, το bounding box, το aspect ratio, το extent και ο αριθμός κορυφών. 
Με βάση αυτά τα χαρακτηριστικά γίνεται απλή rule-based κατηγοριοποίηση τριών αντικειμένων: Book/Notebook, Sunglasses και Box.

Η μέθοδος δεν αποτελεί πλήρες σύστημα αναγνώρισης αντικειμένων με τεχνητή νοημοσύνη, αλλά μία απλή κλασική προσέγγιση υπολογιστικής όρασης που βασίζεται σε ακμές, contours και γεωμετρικούς κανόνες.